# 模块4 第1次课：传统ACE策略回顾 + DeepACE论文精读本notebook包含两部分：1. ACE策略的Python实现回顾与可视化2. DeepACE论文的结构化阅读引导---

## 第一部分：ACE策略回顾### §1 ACE策略概述ACE (Advanced Combination Encoder) 是目前最常用的CI语音编码策略。其核心流程：```输入音频 → 预加重 → 分帧加窗 → FFT → 频带功率包络 → N-of-M通道选择 → 对数压缩 → 电刺激映射```关键参数：- **频带数 (NumberOfBands)**：通常为22，对应CI的22个电极- **Nmaxima**：每帧选取的最大通道数（通常为8-12）- **刺激速率 (StimulationRate)**：每秒刺激脉冲数（通常900-1000 Hz）- **THR / MCL**：阈值/最舒适级别，定义电刺激的动态范围

### §2 运行ACE代码处理一段音频我们使用 `ACE/ace_strategy.py` 来处理一段合成语音，观察ACE的每个处理步骤。

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport sys, os# 添加 ACE 目录到路径sys.path.insert(0, os.path.join('..', 'ACE'))# 生成一段模拟语音（多谐波叠加）sr = 16000duration = 0.5  # 0.5秒t = np.linspace(0, duration, int(sr * duration), endpoint=False)signal = np.zeros_like(t)# 基频 + 谐波f0 = 150  # 基频for h in range(1, 8):    signal += (0.5 / h) * np.sin(2 * np.pi * f0 * h * t)signal += 0.02 * np.random.randn(len(t))  # 微弱噪声signal = signal / np.max(np.abs(signal)) * 0.8print("信号长度:", len(signal))print("采样率:", sr)print("时长:", duration, "秒")plt.figure(figsize=(12, 3))plt.plot(t[:1600], signal[:1600])plt.title("模拟语音信号（前100ms）")plt.xlabel("时间 (s)")plt.ylabel("幅度")plt.tight_layout()plt.show()

In [ ]:
# 运行 ACE 策略处理try:    from ace_strategy import ace_strategy    q, p = ace_strategy(signal, sr, n_band=22, n_maxima=8)    print("ACE 处理完成！")    print("输出字典 q 的键:", list(q.keys()))    print()    print("参数 p 的部分关键参数:")    for key in ['NumberOfBands', 'Nmaxima', 'StimulationRate', 'block_size', 'block_shift']:        if key in p:            print("  %s: %s" % (key, p[key]))    ACE_AVAILABLE = Trueexcept Exception as e:    print("ACE代码运行失败:", e)    print("将使用模拟数据演示")    ACE_AVAILABLE = False

In [ ]:
# 可视化 ACE 输出：电极图（electrodogram）if ACE_AVAILABLE:    electrodes = q['electrodes']    current_levels = q['current_levels']    print("电极序列长度:", len(electrodes))    print("电流级别范围: [%.1f, %.1f]" % (current_levels.min(), current_levels.max()))    print("唯一电极编号:", np.unique(electrodes[electrodes > 0]))    n_bands = p['NumberOfBands']    n_maxima = p['Nmaxima']    n_pulses = len(electrodes)    n_frames = n_pulses // n_maxima    elec_matrix = np.zeros((n_bands, max(n_frames, 1)))    for frame_idx in range(min(n_frames, elec_matrix.shape[1])):        for ch_idx in range(n_maxima):            pulse_idx = frame_idx * n_maxima + ch_idx            if pulse_idx < n_pulses:                elec_num = int(electrodes[pulse_idx])                if 1 <= elec_num <= n_bands:                    elec_matrix[elec_num - 1, frame_idx] = current_levels[pulse_idx]    fig, axes = plt.subplots(1, 2, figsize=(14, 5))    im = axes[0].imshow(elec_matrix, aspect='auto', origin='lower', cmap='hot')    axes[0].set_title("ACE 电极图 (Electrodogram)")    axes[0].set_xlabel("时间帧")    axes[0].set_ylabel("电极通道")    plt.colorbar(im, ax=axes[0], label="电流级别")    if n_frames > 10:        frame_idx = 10        active_channels = elec_matrix[:, frame_idx]        active_mask = active_channels > 0        axes[1].barh(range(n_bands), active_channels)        axes[1].set_title("第%d帧的通道选择 (N-of-M)" % frame_idx)        axes[1].set_xlabel("电流级别")        axes[1].set_ylabel("电极通道")        n_active = np.sum(active_mask)        axes[1].text(0.95, 0.95, "激活通道: %d/%d" % (n_active, n_bands),                    transform=axes[1].transAxes, ha='right', va='top',                    bbox=dict(boxstyle='round', facecolor='wheat'))    plt.tight_layout()    plt.show()else:    # 模拟电极图可视化    n_bands = 22    n_frames = 200    np.random.seed(42)    elec_sim = np.random.rand(n_bands, n_frames) * 200 + 50    for col in range(n_frames):        frame = elec_sim[:, col]        threshold = np.sort(frame)[-8]        frame[frame < threshold] = 0        elec_sim[:, col] = frame    fig, axes = plt.subplots(1, 2, figsize=(14, 5))    im = axes[0].imshow(elec_sim, aspect='auto', origin='lower', cmap='hot')    axes[0].set_title("模拟 ACE 电极图 (Electrodogram)")    axes[0].set_xlabel("时间帧")    axes[0].set_ylabel("电极通道")    plt.colorbar(im, ax=axes[0], label="电流级别")    frame_idx = 50    axes[1].barh(range(n_bands), elec_sim[:, frame_idx])    axes[1].set_title("第%d帧的通道选择 (N-of-M)" % frame_idx)    axes[1].set_xlabel("电流级别")    axes[1].set_ylabel("电极通道")    plt.tight_layout()    plt.show()

### §3 ACE的局限性讨论观察上面的电极图，思考以下问题：1. **固定规则**：N-of-M通道选择使用固定的规则（选幅度最大的N个通道）。这在什么情况下可能不是最优的？2. **无法适应噪声**：当输入是带噪语音时，ACE会怎样？噪声能量可能占据某些通道，导致语音通道被抑制。3. **通道选择策略单一**：ACE只看当前帧的能量，不考虑时序信息。是否可以利用历史帧的信息来做出更好的通道选择？4. **对数压缩参数固定**：THR和MCL是静态的，不随输入变化。是否可以动态调整？> **核心洞察**：传统ACE用**固定规则**处理所有情况，而DeepACE的核心思路是用**数据驱动**的方式学习这些规则。

### §4 实操：对比干净语音与带噪语音的ACE输出观察噪声如何影响ACE的通道选择。

In [ ]:
# 生成带噪语音noise = np.random.randn(len(signal)) * 0.3snr_db = 5  # 5 dB SNRspeech_power = np.mean(signal ** 2)noise_power = np.mean(noise ** 2)snr_linear = 10 ** (snr_db / 10)noise_scaled = noise * np.sqrt(speech_power / (noise_power * snr_linear + 1e-8))noisy_signal = signal + noise_scalednoisy_signal = noisy_signal / np.max(np.abs(noisy_signal)) * 0.8fig, axes = plt.subplots(2, 1, figsize=(12, 4))axes[0].plot(t[:1600], signal[:1600])axes[0].set_title("干净语音")axes[0].set_ylabel("幅度")axes[1].plot(t[:1600], noisy_signal[:1600])axes[1].set_title("带噪语音 (SNR=%d dB)" % snr_db)axes[1].set_xlabel("时间 (s)")axes[1].set_ylabel("幅度")plt.tight_layout()plt.show()# 对比 ACE 处理if ACE_AVAILABLE:    q_clean, _ = ace_strategy(signal, sr, n_band=22, n_maxima=8)    q_noisy, _ = ace_strategy(noisy_signal, sr, n_band=22, n_maxima=8)    elec_clean = q_clean['electrodes']    elec_noisy = q_noisy['electrodes']    min_len = min(len(elec_clean), len(elec_noisy))    diff_ratio = np.mean(elec_clean[:min_len] != elec_noisy[:min_len])    print("通道选择差异比例: %.1f%%" % (diff_ratio * 100))    print()    print("观察：噪声改变了ACE的通道选择模式！")    print("这意味着在噪声环境中，CI用户可能接收到不准确的电刺激模式。")else:    print("(ACE不可用，跳过对比实验)")    print("思考：噪声会如何影响N-of-M通道选择？")    print("提示：噪声通常有较宽的频谱，可能占据多个通道的能量。")

---## 第二部分：DeepACE论文精读### §5 论文阅读方法阅读深度学习论文的推荐步骤：1. **标题 + 摘要**：了解问题和方法的一句话总结2. **引言**：问题定义、现有方法的不足、本文的贡献3. **方法**：网络架构、损失函数、训练数据4. **实验**：数据集、评估指标、对比方法、消融实验5. **结论**：主要发现和局限性> 本notebook不包含论文原文。请在旁边打开论文PDF，按以下引导阅读。

### §6 论文结构化阅读引导卡**第一遍：标题和摘要（5分钟）**回答以下问题：- [ ] DeepACE 要解决什么问题？（提示：CI中的什么环节被DL替代了？）- [ ] 输入是什么？输出是什么？- [ ] 核心创新点用一句话概括？**第二遍：引言（10分钟）**回答以下问题：- [ ] 传统ACE策略有哪些具体局限？论文列了哪几点？- [ ] 为什么用深度学习来解决这些问题是合理的？- [ ] 论文的主要贡献（contributions）有哪几条？**第三遍：方法（20分钟）**回答以下问题：- [ ] 网络架构的编码器（Encoder）做了什么？对应ACE的哪一步？- [ ] Mask Generator 的结构是什么？和模块3学过的什么架构类似？- [ ] 解码器（Decoder）的输出是什么？为什么是22个通道？- [ ] 损失函数是什么？为什么选这个损失？**第四遍：实验（10分钟）**回答以下问题：- [ ] 训练数据是怎么构造的？（重点！这和模型的输入输出直接相关）- [ ] 评估指标有哪些？为什么不用普通的MSE/MAE？- [ ] 和传统ACE对比的结果如何？在什么条件下优势最明显？

### §7 DeepACE 架构概览DeepACE 的整体架构：```输入: 带噪语音 WAV (batch, 1, T_samples)        |        v+-------------+|   Encoder   |  Conv1d(1->N, kernel=L, stride=L/2)|   编码器     |  输出: (batch, N, T')+-------------+        |        v+-------------+|  Rectifier  |  可学习的激活函数（正/负分解）|  整流器     |  输出: (batch, N, T')+-------------+        |        v+--------------------------+|    Mask Generator         |  R 个 Stack x X 个 ConvBlock|    掩码生成器             |  含 SE Layer + LAuReL + Skip Connection|                           |  输出: (batch, M=22, T')  <- 22个电极通道!+--------------------------+        |        v  feats * mask -> masked features        |        v+-------------+|   Decoder   |  ConvTranspose1d(N->M=22)|   解码器     |  输出: (batch, M=22, T')+-------------+        |        v+-------------------+| ChannelRebalancer  |  通道间重新平衡+-------------------+        |        v+-------------+|  Hardtanh   |  输出约束到 [1e-6, 1.0]+-------------+        |        v输出: 电极图 (batch, 22, T')```> 关键理解：DeepACE 不需要显式地做FFT、分帧、频带划分——这些全部被编码器的卷积操作隐式学习了。

### §8 关键创新点与已知知识对照| DeepACE组件 | 对应模块2-3知识 | 创新之处 ||------------|----------------|----------|| Encoder (Conv1d) | 模块2的CNN | 类似语谱图提取，但参数是学习的 || Rectifier | ReLU激活函数 | 将激活函数从固定规则变为可学习参数 || ConvBlock | 模块3的CRNN卷积层 | 加入SE注意力 + LAuReL残差 || Skip Connection | 模块2的ResNet | 和Unet一样的跳跃连接 || Mask (Sigmoid) | 模块3的注意力机制 | 输出22通道掩码，对应22个电极 || ChannelRebalancer | - | CI特有的：平衡通道间刺激水平 |> 思考：为什么输出是22个通道而不是更多或更少？这与CI的物理结构有什么关系？

### §9 训练数据构造方式DeepACE 的训练数据是 **配对数据**：- 干净语音 + 噪声 = 带噪语音 (mixture) [WAV, 16kHz]- 干净语音 -> ACE处理 -> 电极图 (target) [.mat, (T, 22)]**关键设计**：- 输入是**带噪**语音- 目标是对**干净**语音做ACE得到的电极图- 模型需要学习：从带噪语音中恢复出干净语音应有的电极图这和语音增强（模块5）的思路非常类似——都是"从带噪恢复干净"，只是输出域不同。| 对比项 | 语音增强 | DeepACE ||-------|---------|--------|| 输入 | 带噪语音 | 带噪语音 || 输出 | 干净语音的语谱图 | 干净语音的电极图 || 域 | 时频域 | CI刺激域 |

### §10 讨论与思考题1. **方法论思考**：DeepACE的编码器用Conv1d替代了传统的FFT+频带划分。这样做有什么优势？有什么潜在的问题？2. **数据效率**：DeepACE需要配对的(带噪语音, 干净电极图)数据。这种数据获取的成本如何？有没有不需要配对数据的替代方案？3. **因果性 (Causality)**：config.yaml中 `causal: True` 意味着什么？为什么CI处理需要因果性？（提示：实时处理的要求）4. **与ACE的关系**：DeepACE是完全替代ACE，还是在ACE的基础上改进？这两种思路各有什么优劣？5. **泛化性**：DeepACE是在特定说话人和特定噪声上训练的。如果换一个说话人，或遇到训练时没见过的噪声，模型还能正常工作吗？

---## 小结本节课我们：1. 回顾了ACE策略的完整流程，并通过代码运行验证了每个步骤2. 观察了噪声对ACE通道选择的影响，理解了传统方法的局限性3. 按结构化方法精读了DeepACE论文4. 建立了DeepACE架构与已有知识的对应关系**课后任务**：- 完成论文阅读引导卡中的所有问题- 画出DeepACE的完整数据流图，标注每一步的张量形状- 预习下次课：阅读 `DeepACE_torch/` 目录下的代码文件